# Movie Recommender: A Content-Based Approach

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Reading and cleaning data

## Movie Dataset

In [2]:
# We will read the dataset in chunks to find modern movies
chunk_size = 100000
modern_movies_list = []

# Read title.basics.tsv.gz in chunks
for chunk in pd.read_csv('https://datasets.imdbws.com/title.basics.tsv.gz', sep='\t', compression='gzip', low_memory=False, chunksize=chunk_size):
    # Clean the startYear column
    chunk['startYear'] = chunk['startYear'].replace('\\N', np.nan)
    chunk['startYear'] = pd.to_numeric(chunk['startYear'], errors='coerce')

    # Filter for movies between 2000 and 2026
    filtered_chunk = chunk[(chunk['startYear'] >= 2000) & (chunk['startYear'] <= 2026) & (chunk['titleType'] == 'movie')]
    modern_movies_list.append(filtered_chunk)

    # Stop after finding a solid pool of modern movies (e.g., 20,000 movies) to keep execution fast
    if sum(len(df) for df in modern_movies_list) >= 20000:
        break

movie_df = pd.concat(modern_movies_list, ignore_index=True)
movie_df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0011801,movie,Tötet nicht mehr,Tötet nicht mehr,0,2019.0,\N,\N,"Action,Crime"
1,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,\N,118,"Comedy,Fantasy,Romance"
2,tt0046976,movie,Il figlio dell'uomo,Il figlio dell'uomo,0,2000.0,\N,\N,\N
3,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,0,2020.0,\N,70,"Drama,Mystery"
4,tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,0,2022.0,\N,85,Documentary
...,...,...,...,...,...,...,...,...,...
26909,tt0499606,movie,Price to Pay,Price to Pay,0,2006.0,\N,\N,Drama
26910,tt0499608,movie,Tranced,Tranced,0,2010.0,\N,\N,"Drama,Fantasy,Romance"
26911,tt0499626,movie,Digital Broadcast,Digital Broadcast,0,2006.0,\N,50,Documentary
26912,tt0499670,movie,El traductor,El traductor,0,2007.0,\N,92,\N


In [3]:
columns_to_replace = ['startYear', 'endYear', 'runtimeMinutes', 'genres']
movie_df[columns_to_replace] = movie_df[columns_to_replace].replace('\\N', np.nan)

movie_df

/tmp/ipykernel_1426/1322524427.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  movie_df[columns_to_replace] = movie_df[columns_to_replace].replace('\\N', np.nan)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0011801,movie,Tötet nicht mehr,Tötet nicht mehr,0,2019.0,NaN,NaN,"Action,Crime"
1,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,NaN,118,"Comedy,Fantasy,Romance"
2,tt0046976,movie,Il figlio dell'uomo,Il figlio dell'uomo,0,2000.0,NaN,NaN,NaN
3,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,0,2020.0,NaN,70,"Drama,Mystery"
4,tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,0,2022.0,NaN,85,Documentary
...,...,...,...,...,...,...,...,...,...
26909,tt0499606,movie,Price to Pay,Price to Pay,0,2006.0,NaN,NaN,Drama
26910,tt0499608,movie,Tranced,Tranced,0,2010.0,NaN,NaN,"Drama,Fantasy,Romance"
26911,tt0499626,movie,Digital Broadcast,Digital Broadcast,0,2006.0,NaN,50,Documentary
26912,tt0499670,movie,El traductor,El traductor,0,2007.0,NaN,92,NaN


In [4]:
print(movie_df.isnull().sum())

tconst                0
titleType             0
primaryTitle          0
originalTitle         0
isAdult               0
startYear             0
endYear           26914
runtimeMinutes     3214
genres             1814
dtype: int64


In [5]:
movie_df['endYear'] = movie_df['endYear'].fillna(movie_df['startYear'])
movie_df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0011801,movie,Tötet nicht mehr,Tötet nicht mehr,0,2019.0,2019.0,NaN,"Action,Crime"
1,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,2001.0,118,"Comedy,Fantasy,Romance"
2,tt0046976,movie,Il figlio dell'uomo,Il figlio dell'uomo,0,2000.0,2000.0,NaN,NaN
3,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,0,2020.0,2020.0,70,"Drama,Mystery"
4,tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,0,2022.0,2022.0,85,Documentary
...,...,...,...,...,...,...,...,...,...
26909,tt0499606,movie,Price to Pay,Price to Pay,0,2006.0,2006.0,NaN,Drama
26910,tt0499608,movie,Tranced,Tranced,0,2010.0,2010.0,NaN,"Drama,Fantasy,Romance"
26911,tt0499626,movie,Digital Broadcast,Digital Broadcast,0,2006.0,2006.0,50,Documentary
26912,tt0499670,movie,El traductor,El traductor,0,2007.0,2007.0,92,NaN


In [6]:
imputer = KNNImputer(n_neighbors=5)
movie_df.loc[:, ['runtimeMinutes']] = imputer.fit_transform(movie_df[['runtimeMinutes']]).astype(int)
movie_df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0011801,movie,Tötet nicht mehr,Tötet nicht mehr,0,2019.0,2019.0,94,"Action,Crime"
1,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,2001.0,118,"Comedy,Fantasy,Romance"
2,tt0046976,movie,Il figlio dell'uomo,Il figlio dell'uomo,0,2000.0,2000.0,94,NaN
3,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,0,2020.0,2020.0,70,"Drama,Mystery"
4,tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,0,2022.0,2022.0,85,Documentary
...,...,...,...,...,...,...,...,...,...
26909,tt0499606,movie,Price to Pay,Price to Pay,0,2006.0,2006.0,94,Drama
26910,tt0499608,movie,Tranced,Tranced,0,2010.0,2010.0,94,"Drama,Fantasy,Romance"
26911,tt0499626,movie,Digital Broadcast,Digital Broadcast,0,2006.0,2006.0,50,Documentary
26912,tt0499670,movie,El traductor,El traductor,0,2007.0,2007.0,92,NaN


In [7]:
movie_df['genres'] = movie_df['genres'].fillna('Unknown')
movie_df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0011801,movie,Tötet nicht mehr,Tötet nicht mehr,0,2019.0,2019.0,94,"Action,Crime"
1,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,2001.0,118,"Comedy,Fantasy,Romance"
2,tt0046976,movie,Il figlio dell'uomo,Il figlio dell'uomo,0,2000.0,2000.0,94,Unknown
3,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,0,2020.0,2020.0,70,"Drama,Mystery"
4,tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,0,2022.0,2022.0,85,Documentary
...,...,...,...,...,...,...,...,...,...
26909,tt0499606,movie,Price to Pay,Price to Pay,0,2006.0,2006.0,94,Drama
26910,tt0499608,movie,Tranced,Tranced,0,2010.0,2010.0,94,"Drama,Fantasy,Romance"
26911,tt0499626,movie,Digital Broadcast,Digital Broadcast,0,2006.0,2006.0,50,Documentary
26912,tt0499670,movie,El traductor,El traductor,0,2007.0,2007.0,92,Unknown


In [8]:
movie_df['runtimeHours'] = movie_df['runtimeMinutes'] // 60
movie_df['runtimeMinutes'] = movie_df['runtimeMinutes'] % 60

def format_runtime(row):
    if row['runtimeHours'] == 0:
        return f"{row['runtimeMinutes']} min"
    elif row['runtimeMinutes'] == 0:
        return f"{row['runtimeHours']} h"
    else:
        return f"{row['runtimeHours']} h {row['runtimeMinutes']} min"

movie_df['runtime'] = movie_df.apply(format_runtime, axis=1)
movie_df = movie_df.drop(['runtimeMinutes', 'runtimeHours'], axis=1)
movie_df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,genres,runtime
0,tt0011801,movie,Tötet nicht mehr,Tötet nicht mehr,0,2019.0,2019.0,"Action,Crime",1 h 34 min
1,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,2001.0,"Comedy,Fantasy,Romance",1 h 58 min
2,tt0046976,movie,Il figlio dell'uomo,Il figlio dell'uomo,0,2000.0,2000.0,Unknown,1 h 34 min
3,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,0,2020.0,2020.0,"Drama,Mystery",1 h 10 min
4,tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,0,2022.0,2022.0,Documentary,1 h 25 min
...,...,...,...,...,...,...,...,...,...
26909,tt0499606,movie,Price to Pay,Price to Pay,0,2006.0,2006.0,Drama,1 h 34 min
26910,tt0499608,movie,Tranced,Tranced,0,2010.0,2010.0,"Drama,Fantasy,Romance",1 h 34 min
26911,tt0499626,movie,Digital Broadcast,Digital Broadcast,0,2006.0,2006.0,Documentary,50 min
26912,tt0499670,movie,El traductor,El traductor,0,2007.0,2007.0,Unknown,1 h 32 min


In [9]:
def transform_to_list(x):
    if x == '\\N':
        return []
    return x.split(',') if ',' in x else [x]

movie_df['genres'] = movie_df['genres'].apply(transform_to_list)
movie_df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,genres,runtime
0,tt0011801,movie,Tötet nicht mehr,Tötet nicht mehr,0,2019.0,2019.0,"[Action, Crime]",1 h 34 min
1,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,2001.0,"[Comedy, Fantasy, Romance]",1 h 58 min
2,tt0046976,movie,Il figlio dell'uomo,Il figlio dell'uomo,0,2000.0,2000.0,[Unknown],1 h 34 min
3,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,0,2020.0,2020.0,"[Drama, Mystery]",1 h 10 min
4,tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,0,2022.0,2022.0,[Documentary],1 h 25 min
...,...,...,...,...,...,...,...,...,...
26909,tt0499606,movie,Price to Pay,Price to Pay,0,2006.0,2006.0,[Drama],1 h 34 min
26910,tt0499608,movie,Tranced,Tranced,0,2010.0,2010.0,"[Drama, Fantasy, Romance]",1 h 34 min
26911,tt0499626,movie,Digital Broadcast,Digital Broadcast,0,2006.0,2006.0,[Documentary],50 min
26912,tt0499670,movie,El traductor,El traductor,0,2007.0,2007.0,[Unknown],1 h 32 min


In [10]:
print(movie_df.isnull().sum())

tconst           0
titleType        0
primaryTitle     0
originalTitle    0
isAdult          0
startYear        0
endYear          0
genres           0
runtime          0
dtype: int64


## Rating Dataset

In [11]:
# Get the set of movie IDs (tconst) we already filtered in movie_df
valid_tconsts = set(movie_df['tconst'])

# Read the ratings file in chunks and only keep ratings for our movies
ratings_list = []
for chunk in pd.read_csv('https://datasets.imdbws.com/title.ratings.tsv.gz', sep='\t', compression='gzip', low_memory=False, chunksize=100000):
    # Keep only rows where tconst is in our valid_tconsts set
    filtered_chunk = chunk[chunk['tconst'].isin(valid_tconsts)]
    ratings_list.append(filtered_chunk)

# Combine into rating_df
rating_df = pd.concat(ratings_list, ignore_index=True)
rating_df

,tconst,averageRating,numVotes
0,tt0035423,6.4,94995
1,tt0062336,6.3,299
2,tt0065414,6.9,41
3,tt0069049,6.6,8645
4,tt0070596,6.8,245
...,...,...,...
23041,tt0499603,6.5,17291
23042,tt0499604,7.6,28
23043,tt0499606,2.4,302
23044,tt0499608,5.6,69


In [12]:
print(rating_df.isnull().sum())

tconst           0
averageRating    0
numVotes         0
dtype: int64


## Movie Rating Dataset

In [13]:
movie_rating_df = pd.merge(movie_df, rating_df, on='tconst', how='inner')
movie_rating_df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,genres,runtime,averageRating,numVotes
0,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,2001.0,"[Comedy, Fantasy, Romance]",1 h 58 min,6.4,94995
1,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,0,2020.0,2020.0,"[Drama, Mystery]",1 h 10 min,6.3,299
2,tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,0,2022.0,2022.0,[Documentary],1 h 25 min,6.9,41
3,tt0069049,movie,The Other Side of the Wind,The Other Side of the Wind,0,2018.0,2018.0,[Drama],2 h 2 min,6.6,8645
4,tt0070596,movie,Socialist Realism,El realismo socialista,0,2023.0,2023.0,"[Drama, History]",1 h 18 min,6.8,245
...,...,...,...,...,...,...,...,...,...,...,...
23041,tt0499603,movie,10 Items or Less,10 Items or Less,0,2006.0,2006.0,"[Comedy, Drama]",1 h 22 min,6.5,17291
23042,tt0499604,movie,Changing Spots,Changing Spots,0,2006.0,2006.0,[Drama],1 h 30 min,7.6,28
23043,tt0499606,movie,Price to Pay,Price to Pay,0,2006.0,2006.0,[Drama],1 h 34 min,2.4,302
23044,tt0499608,movie,Tranced,Tranced,0,2010.0,2010.0,"[Drama, Fantasy, Romance]",1 h 34 min,5.6,69


## Weighted Rating

In [14]:
C = movie_rating_df['averageRating'].mean()
print(C)

6.10796233619717


In [15]:
m = movie_rating_df['numVotes'].quantile(0.8)
print(m)

1585.0


In [16]:
def imdb_weighted_rating(df, var=0.8):
    v = df['numVotes']
    R = df['averageRating']
    C = df['averageRating'].mean()
    m = df['numVotes'].quantile(var)
    df['score'] = (v/(m+v))*R + (m/(m+v))* C
    return df['score']

movie_rating_df['score'] = imdb_weighted_rating(movie_rating_df)
movie_rating_df['score'] = movie_rating_df['score'].round(2)
movie_rating_df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,genres,runtime,averageRating,numVotes,score
0,tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,2001.0,"[Comedy, Fantasy, Romance]",1 h 58 min,6.4,94995,6.40
1,tt0062336,movie,The Tango of the Widower and Its Distorting Mi...,El tango del viudo y su espejo deformante,0,2020.0,2020.0,"[Drama, Mystery]",1 h 10 min,6.3,299,6.14
2,tt0065414,movie,Appunti per un romanzo sull'immondezza,Appunti per un romanzo sull'immondezza,0,2022.0,2022.0,[Documentary],1 h 25 min,6.9,41,6.13
3,tt0069049,movie,The Other Side of the Wind,The Other Side of the Wind,0,2018.0,2018.0,[Drama],2 h 2 min,6.6,8645,6.52
4,tt0070596,movie,Socialist Realism,El realismo socialista,0,2023.0,2023.0,"[Drama, History]",1 h 18 min,6.8,245,6.20
...,...,...,...,...,...,...,...,...,...,...,...,...
23041,tt0499603,movie,10 Items or Less,10 Items or Less,0,2006.0,2006.0,"[Comedy, Drama]",1 h 22 min,6.5,17291,6.47
23042,tt0499604,movie,Changing Spots,Changing Spots,0,2006.0,2006.0,[Drama],1 h 30 min,7.6,28,6.13
23043,tt0499606,movie,Price to Pay,Price to Pay,0,2006.0,2006.0,[Drama],1 h 34 min,2.4,302,5.51
23044,tt0499608,movie,Tranced,Tranced,0,2010.0,2010.0,"[Drama, Fantasy, Romance]",1 h 34 min,5.6,69,6.09


## Director Dataset

In [17]:
director_list = []
for chunk in pd.read_csv('https://datasets.imdbws.com/title.crew.tsv.gz', sep='\t', compression='gzip', low_memory=False, chunksize=100000):
    columns_to_replace = ['directors', 'writers']
    chunk[columns_to_replace] = chunk[columns_to_replace].replace('\\N', np.nan)
    filtered_chunk = chunk[chunk['tconst'].isin(valid_tconsts)]
    director_list.append(filtered_chunk)

director_df = pd.concat(director_list, ignore_index=True)
director_df

,tconst,directors,writers
0,tt0011801,nm0681726,"nm0483944,nm0681726"
1,tt0035423,nm0003506,"nm0737216,nm0003506"
2,tt0046976,nm0754670,"nm5461783,nm0274767"
3,tt0062336,nm0749914,"nm0749914,nm1146177"
4,tt0065414,nm0001596,NaN
...,...,...,...
26909,tt0499606,nm1083613,nm1083613
26910,tt0499608,nm0262693,nm2115119
26911,tt0499626,nm1807599,nm1825191
26912,tt0499670,nm0868398,nm0868398


In [18]:
def transform_to_list(x):
    if pd.isna(x) or x == '\\N':
        return []
    return x.split(',') if ',' in str(x) else [x]

director_df['directors'] = director_df['directors'].apply(transform_to_list)
director_df['writers'] = director_df['writers'].apply(transform_to_list)
director_df

,tconst,directors,writers
0,tt0011801,[nm0681726],"[nm0483944, nm0681726]"
1,tt0035423,[nm0003506],"[nm0737216, nm0003506]"
2,tt0046976,[nm0754670],"[nm5461783, nm0274767]"
3,tt0062336,[nm0749914],"[nm0749914, nm1146177]"
4,tt0065414,[nm0001596],[]
...,...,...,...
26909,tt0499606,[nm1083613],[nm1083613]
26910,tt0499608,[nm0262693],[nm2115119]
26911,tt0499626,[nm1807599],[nm1825191]
26912,tt0499670,[nm0868398],[nm0868398]


In [19]:
needed_nconsts = set()
for lst in director_df['directors'].dropna():
    needed_nconsts.update(lst)
for lst in director_df['writers'].dropna():
    needed_nconsts.update(lst)

names_mapped_list = []

for chunk in pd.read_csv('https://datasets.imdbws.com/name.basics.tsv.gz', sep='\t', compression='gzip', low_memory=False, chunksize=150000):
    chunk['primaryProfession'] = chunk['primaryProfession'].fillna('')
    is_actor = chunk['primaryProfession'].str.contains('actor|actress', case=False)
    is_needed = chunk['nconst'].isin(needed_nconsts)

    filtered_chunk = chunk[is_actor | is_needed]
    names_mapped_list.append(filtered_chunk[['nconst', 'primaryName']])

    if sum(len(d) for d in names_mapped_list) >= 120000:
        break

global_names_df = pd.concat(names_mapped_list, ignore_index=True)
global_name_mapping = dict(zip(global_names_df['nconst'], global_names_df['primaryName']))

global_names_df

,nconst,primaryName
0,nm0000001,Fred Astaire
1,nm0000002,Lauren Bacall
2,nm0000003,Brigitte Bardot
3,nm0000004,John Belushi
4,nm0000005,Ingmar Bergman
...,...,...
177066,nm0317352,Ramón Gieling
177067,nm0317356,Mieczyslaw Gielniewski
177068,nm0317360,Gary Giem
177069,nm0317361,Pamela Gien


In [20]:
missing_nconsts = set()
for col in ['directors', 'writers']:
    for lst in director_df[col]:
        if isinstance(lst, list):
            for code in lst:
                if code.startswith('nm') and code not in global_name_mapping:
                    missing_nconsts.add(code)

if len(missing_nconsts) > 0:
    extra_mapped = []
    for chunk in pd.read_csv('https://datasets.imdbws.com/name.basics.tsv.gz', sep='\t', compression='gzip', low_memory=False, chunksize=150000):
        filtered_chunk = chunk[chunk['nconst'].isin(missing_nconsts)]
        if not filtered_chunk.empty:
            extra_mapped.append(filtered_chunk[['nconst', 'primaryName']])
            missing_nconsts -= set(filtered_chunk['nconst'])
        if len(missing_nconsts) == 0:
            break

    if extra_mapped:
        extra_df = pd.concat(extra_mapped, ignore_index=True)
        global_name_mapping.update(dict(zip(extra_df['nconst'], extra_df['primaryName'])))

def replace_with_names(name_list):
    if isinstance(name_list, list):
        return [global_name_mapping.get(code, code) for code in name_list]
    return name_list

director_df['directors'] = director_df['directors'].apply(replace_with_names)
director_df['writers'] = director_df['writers'].apply(replace_with_names)

director_df

,tconst,directors,writers
0,tt0011801,[Lupu Pick],"[Gerhard Lamprecht, Lupu Pick]"
1,tt0035423,[James Mangold],"[Steven Rogers, James Mangold]"
2,tt0046976,[Virgilio Sabel],"[Giacomo Alberione, Liana Ferri]"
3,tt0062336,[Raúl Ruiz],"[Raúl Ruiz, Omar Saavedra Santis]"
4,tt0065414,[Pier Paolo Pasolini],[]
...,...,...,...
26909,tt0499606,[Michael McCready],[Michael McCready]
26910,tt0499608,[David Mickey Evans],[Marshall Sylver]
26911,tt0499626,[Nick Peterson],[Nicolas Gaud]
26912,tt0499670,[Oliverio Torre],[Oliverio Torre]


In [21]:
print(director_df.isnull().sum())

tconst       0
directors    0
writers      0
dtype: int64


## Actor Dataset

In [22]:
actors_list = []
for chunk in pd.read_csv('https://datasets.imdbws.com/name.basics.tsv.gz', sep='\t', compression='gzip', low_memory=False, chunksize=150000, na_values='\\N'):
    chunk['primaryProfession'] = chunk['primaryProfession'].fillna('')
    is_actor = chunk['primaryProfession'].str.contains('actor|actress', case=False)
    is_needed = chunk['nconst'].isin(needed_nconsts)

    filtered_chunk = chunk[is_actor | is_needed]
    actors_list.append(filtered_chunk[['nconst', 'primaryName', 'knownForTitles']])

    if sum(len(d) for d in actors_list) >= 120000:
        break

raw_actor_df = pd.concat(actors_list, ignore_index=True)
raw_actor_df = raw_actor_df.dropna(subset=['knownForTitles'])
raw_actor_df

,nconst,primaryName,knownForTitles
0,nm0000001,Fred Astaire,"tt0072308,tt0031983,tt0050419,tt0025164"
1,nm0000002,Lauren Bacall,"tt0037382,tt0117057,tt0075213,tt0038355"
2,nm0000003,Brigitte Bardot,"tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,"tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000005,Ingmar Bergman,"tt0050986,tt0069467,tt0083922,tt0050976"
...,...,...,...
177066,nm0317352,Ramón Gieling,"tt0263381,tt4362388,tt2160325,tt7603218"
177067,nm0317356,Mieczyslaw Gielniewski,"tt0022947,tt0027897,tt1042397,tt0953903"
177068,nm0317360,Gary Giem,"tt0119210,tt0069557,tt0096582,tt0069658"
177069,nm0317361,Pamela Gien,"tt0392812,tt0219919,tt0096708,tt0086734"


In [23]:
raw_actor_df['knownForTitles'].apply(lambda x: len(x.split(','))).unique()
raw_actor_df['knownForTitles'] = raw_actor_df['knownForTitles'].apply(lambda x: x.split(','))
raw_actor_df

,nconst,primaryName,knownForTitles
0,nm0000001,Fred Astaire,"[tt0072308, tt0031983, tt0050419, tt0025164]"
1,nm0000002,Lauren Bacall,"[tt0037382, tt0117057, tt0075213, tt0038355]"
2,nm0000003,Brigitte Bardot,"[tt0057345, tt0049189, tt0056404, tt0054452]"
3,nm0000004,John Belushi,"[tt0072562, tt0077975, tt0080455, tt0078723]"
4,nm0000005,Ingmar Bergman,"[tt0050986, tt0069467, tt0083922, tt0050976]"
...,...,...,...
177066,nm0317352,Ramón Gieling,"[tt0263381, tt4362388, tt2160325, tt7603218]"
177067,nm0317356,Mieczyslaw Gielniewski,"[tt0022947, tt0027897, tt1042397, tt0953903]"
177068,nm0317360,Gary Giem,"[tt0119210, tt0069557, tt0096582, tt0069658]"
177069,nm0317361,Pamela Gien,"[tt0392812, tt0219919, tt0096708, tt0086734]"


## Joining Datasets

In [24]:
unnested_df = raw_actor_df.explode('knownForTitles')
unnested_df

,nconst,primaryName,knownForTitles
0,nm0000001,Fred Astaire,tt0072308
0,nm0000001,Fred Astaire,tt0031983
0,nm0000001,Fred Astaire,tt0050419
0,nm0000001,Fred Astaire,tt0025164
1,nm0000002,Lauren Bacall,tt0037382
...,...,...,...
177069,nm0317361,Pamela Gien,tt0086734
177070,nm0317364,Michael Gier,tt1378167
177070,nm0317364,Michael Gier,tt21876766
177070,nm0317364,Michael Gier,tt9256574


In [26]:
unnested_drop = unnested_df.drop(['nconst'], axis=1)
df_grouped = unnested_drop.groupby('knownForTitles')['primaryName'].agg(list).reset_index()
df_grouped.columns = ['knownForTitles', 'cast_name']
df_grouped

,knownForTitles,cast_name
0,tt0000007,[Peter Courtney]
1,tt0000009,"[Blanche Bayliss, William Courtenay]"
2,tt0000014,"[François Clerc, Benoît Duval]"
3,tt0000075,[Jehanne d'Alcy]
4,tt0000165,"[José Amigo, Antonio Fino]"
...,...,...
200884,tt9914574,[Louann Fernald]
200885,tt9915010,[Lolly Boroff]
200886,tt9915710,[Gabrielle]
200887,tt9916124,[Pedro Alcántara]


In [27]:
base_df = pd.merge(df_grouped, movie_rating_df, left_on='knownForTitles', right_on='tconst', how='inner')
base_df = pd.merge(base_df, director_df, left_on='tconst', right_on='tconst', how='left')
base_df

,knownForTitles,cast_name,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,genres,runtime,averageRating,numVotes,score,directors,writers
0,tt0035423,"[Charlotte Ayanna, Vic Bucossi, Cornelius Patr...",tt0035423,movie,Kate & Leopold,Kate & Leopold,0,2001.0,2001.0,"[Comedy, Fantasy, Romance]",1 h 58 min,6.4,94995,6.40,[James Mangold],"[Steven Rogers, James Mangold]"
1,tt0069049,"[Erik Berchot, Ron Charles, Daniel Ciampolini,...",tt0069049,movie,The Other Side of the Wind,The Other Side of the Wind,0,2018.0,2018.0,[Drama],2 h 2 min,6.6,8645,6.52,[Orson Welles],"[Orson Welles, Oja Kodar]"
2,tt0080065,"[Deborah Au Luce, Susan Clark, Lance Garrett, ...",tt0080065,movie,Up River,Up River,0,2026.0,2026.0,"[Drama, Western]",1 h 26 min,4.3,86,6.01,[Carl Kitt],"[Don Clark, Susan Clark, Carol Hummel]"
3,tt0088751,"[Debra Berwick, Wayne Berwick, Tina Cheri, Rob...",tt0088751,movie,The Naked Monster,The Naked Monster,0,2005.0,2005.0,"[Comedy, Horror, Sci-Fi]",1 h 40 min,5.3,391,5.95,"[Wayne Berwick, Ted Newsom]",[Ted Newsom]
4,tt0096056,[Avital Dicker],tt0096056,movie,Crime and Punishment,Crime and Punishment,0,2002.0,2002.0,[Drama],2 h 6 min,5.4,983,5.84,[Menahem Golan],"[Fyodor Dostoevsky, Menahem Golan]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11252,tt0499556,"[Kerry Phillips, John K. Alexander, Laura Arca...",tt0499556,movie,War,War,0,2007.0,2007.0,"[Action, Thriller]",1 h 43 min,6.2,102364,6.20,[Philip G. Atwell],"[Lee Anthony Smith, Gregory J. Bradley]"
11253,tt0499586,[Maria Antoniou],tt0499586,movie,Brothers: The Return,Tusenbröder - Återkomsten,0,2006.0,2006.0,"[Crime, Drama, Thriller]",1 h 36 min,6.6,1086,6.31,[Erik Leijonborg],[Lars Lundström]
11254,tt0499596,[William Arntz],tt0499596,movie,What the Bleep!?: Down the Rabbit Hole,What the Bleep!?: Down the Rabbit Hole,0,2006.0,2006.0,"[Documentary, Drama]",2 h 36 min,6.4,3074,6.30,"[William Arntz, Betsy Chasse, Mark Vicente]",[]
11255,tt0499604,[Natalie Anderson],tt0499604,movie,Changing Spots,Changing Spots,0,2006.0,2006.0,[Drama],1 h 30 min,7.6,28,6.13,[Susan Turley],"[Susan Turley, Lane West]"


In [28]:
base_drop = base_df.drop(['knownForTitles'], axis=1)
print(base_drop.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11257 entries, 0 to 11256
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   cast_name      11257 non-null  object 
 1   tconst         11257 non-null  object 
 2   titleType      11257 non-null  object 
 3   primaryTitle   11257 non-null  object 
 4   originalTitle  11257 non-null  object 
 5   isAdult        11257 non-null  int64  
 6   startYear      11257 non-null  float64
 7   endYear        11257 non-null  float64
 8   genres         11257 non-null  object 
 9   runtime        11257 non-null  object 
 10  averageRating  11257 non-null  float64
 11  numVotes       11257 non-null  int64  
 12  score          11257 non-null  float64
 13  directors      11257 non-null  object 
 14  writers        11257 non-null  object 
dtypes: float64(4), int64(2), object(9)
memory usage: 1.3+ MB
None


In [29]:
base_drop[['directors', 'writers']] = base_drop[['directors', 'writers']].fillna('unknown')

In [30]:
print(base_drop.isnull().sum())

cast_name        0
tconst           0
titleType        0
primaryTitle     0
originalTitle    0
isAdult          0
startYear        0
endYear          0
genres           0
runtime          0
averageRating    0
numVotes         0
score            0
directors        0
writers          0
dtype: int64


In [31]:
base_drop = base_drop[['primaryTitle','titleType','startYear','runtime','genres','averageRating', 'score', 'numVotes','cast_name','directors','writers']]
base_drop.columns = ['title','type','start','duration','genres','rating','votes', 'score', 'cast_name','directors','writers']
base_drop

,title,type,start,duration,genres,rating,votes,score,cast_name,directors,writers
0,Kate & Leopold,movie,2001.0,1 h 58 min,"[Comedy, Fantasy, Romance]",6.4,6.40,94995,"[Charlotte Ayanna, Vic Bucossi, Cornelius Patr...",[James Mangold],"[Steven Rogers, James Mangold]"
1,The Other Side of the Wind,movie,2018.0,2 h 2 min,[Drama],6.6,6.52,8645,"[Erik Berchot, Ron Charles, Daniel Ciampolini,...",[Orson Welles],"[Orson Welles, Oja Kodar]"
2,Up River,movie,2026.0,1 h 26 min,"[Drama, Western]",4.3,6.01,86,"[Deborah Au Luce, Susan Clark, Lance Garrett, ...",[Carl Kitt],"[Don Clark, Susan Clark, Carol Hummel]"
3,The Naked Monster,movie,2005.0,1 h 40 min,"[Comedy, Horror, Sci-Fi]",5.3,5.95,391,"[Debra Berwick, Wayne Berwick, Tina Cheri, Rob...","[Wayne Berwick, Ted Newsom]",[Ted Newsom]
4,Crime and Punishment,movie,2002.0,2 h 6 min,[Drama],5.4,5.84,983,[Avital Dicker],[Menahem Golan],"[Fyodor Dostoevsky, Menahem Golan]"
...,...,...,...,...,...,...,...,...,...,...,...
11252,War,movie,2007.0,1 h 43 min,"[Action, Thriller]",6.2,6.20,102364,"[Kerry Phillips, John K. Alexander, Laura Arca...",[Philip G. Atwell],"[Lee Anthony Smith, Gregory J. Bradley]"
11253,Brothers: The Return,movie,2006.0,1 h 36 min,"[Crime, Drama, Thriller]",6.6,6.31,1086,[Maria Antoniou],[Erik Leijonborg],[Lars Lundström]
11254,What the Bleep!?: Down the Rabbit Hole,movie,2006.0,2 h 36 min,"[Documentary, Drama]",6.4,6.30,3074,[William Arntz],"[William Arntz, Betsy Chasse, Mark Vicente]",[]
11255,Changing Spots,movie,2006.0,1 h 30 min,[Drama],7.6,6.13,28,[Natalie Anderson],[Susan Turley],"[Susan Turley, Lane West]"


## Feature Engineering

In [32]:
feature_df = base_drop[['title','cast_name','genres','directors','writers', 'rating', 'votes', 'score']].copy()
feature_df['cast_name'] = feature_df['cast_name'].apply(lambda x: x[:3] if isinstance(x, list) else [])
feature_df

,title,cast_name,genres,directors,writers,rating,votes,score
0,Kate & Leopold,"[Charlotte Ayanna, Vic Bucossi, Cornelius Patr...","[Comedy, Fantasy, Romance]",[James Mangold],"[Steven Rogers, James Mangold]",6.4,6.40,94995
1,The Other Side of the Wind,"[Erik Berchot, Ron Charles, Daniel Ciampolini,...",[Drama],[Orson Welles],"[Orson Welles, Oja Kodar]",6.6,6.52,8645
2,Up River,"[Deborah Au Luce, Susan Clark, Lance Garrett, ...","[Drama, Western]",[Carl Kitt],"[Don Clark, Susan Clark, Carol Hummel]",4.3,6.01,86
3,The Naked Monster,"[Debra Berwick, Wayne Berwick, Tina Cheri, Rob...","[Comedy, Horror, Sci-Fi]","[Wayne Berwick, Ted Newsom]",[Ted Newsom],5.3,5.95,391
4,Crime and Punishment,[Avital Dicker],[Drama],[Menahem Golan],"[Fyodor Dostoevsky, Menahem Golan]",5.4,5.84,983
...,...,...,...,...,...,...,...,...
11252,War,"[Kerry Phillips, John K. Alexander, Laura Arca...","[Action, Thriller]",[Philip G. Atwell],"[Lee Anthony Smith, Gregory J. Bradley]",6.2,6.20,102364
11253,Brothers: The Return,[Maria Antoniou],"[Crime, Drama, Thriller]",[Erik Leijonborg],[Lars Lundström],6.6,6.31,1086
11254,What the Bleep!?: Down the Rabbit Hole,[William Arntz],"[Documentary, Drama]","[William Arntz, Betsy Chasse, Mark Vicente]",[],6.4,6.30,3074
11255,Changing Spots,[Natalie Anderson],[Drama],[Susan Turley],"[Susan Turley, Lane West]",7.6,6.13,28


In [34]:
def sanitize(x):
    if isinstance(x, list):
        return [i.replace(' ', '').lower() for i in x]
    return []

feature_df['soup'] = feature_df.apply(lambda row:
    ' '.join(sanitize(row['cast_name'])) + ' ' +
    ' '.join(sanitize(row['genres'])) + ' ' +
    ' '.join(sanitize(row['directors'])) + ' ' +
    ' '.join(sanitize(row['writers'])),
    axis=1
)

feature_df

,title,cast_name,genres,directors,writers,rating,votes,score,soup
0,Kate & Leopold,"[Charlotte Ayanna, Vic Bucossi, Cornelius Patr...","[Comedy, Fantasy, Romance]",[James Mangold],"[Steven Rogers, James Mangold]",6.4,6.40,94995,charlotteayanna vicbucossi corneliuspatrickbyr...
1,The Other Side of the Wind,"[Erik Berchot, Ron Charles, Daniel Ciampolini]",[Drama],[Orson Welles],"[Orson Welles, Oja Kodar]",6.6,6.52,8645,erikberchot roncharles danielciampolini drama ...
2,Up River,"[Deborah Au Luce, Susan Clark, Lance Garrett]","[Drama, Western]",[Carl Kitt],"[Don Clark, Susan Clark, Carol Hummel]",4.3,6.01,86,deborahauluce susanclark lancegarrett drama we...
3,The Naked Monster,"[Debra Berwick, Wayne Berwick, Tina Cheri]","[Comedy, Horror, Sci-Fi]","[Wayne Berwick, Ted Newsom]",[Ted Newsom],5.3,5.95,391,debraberwick wayneberwick tinacheri comedy hor...
4,Crime and Punishment,[Avital Dicker],[Drama],[Menahem Golan],"[Fyodor Dostoevsky, Menahem Golan]",5.4,5.84,983,avitaldicker drama menahemgolan fyodordostoevs...
...,...,...,...,...,...,...,...,...,...
11252,War,"[Kerry Phillips, John K. Alexander, Laura Arca...","[Action, Thriller]",[Philip G. Atwell],"[Lee Anthony Smith, Gregory J. Bradley]",6.2,6.20,102364,kerryphillips johnk.alexander lauraarcangeli a...
11253,Brothers: The Return,[Maria Antoniou],"[Crime, Drama, Thriller]",[Erik Leijonborg],[Lars Lundström],6.6,6.31,1086,mariaantoniou crime drama thriller erikleijonb...
11254,What the Bleep!?: Down the Rabbit Hole,[William Arntz],"[Documentary, Drama]","[William Arntz, Betsy Chasse, Mark Vicente]",[],6.4,6.30,3074,williamarntz documentary drama williamarntz be...
11255,Changing Spots,[Natalie Anderson],[Drama],[Susan Turley],"[Susan Turley, Lane West]",7.6,6.13,28,natalieanderson drama susanturley susanturley ...


## Modeling and Evaluation

In [35]:
count = CountVectorizer()
count_matrix = count.fit_transform(feature_df['soup'])

print(count)
print("Dimensi matriks:", count_matrix.shape)

CountVectorizer()
Dimensi matriks: (11257, 34430)


In [36]:
cosine_sim = cosine_similarity(count_matrix, count_matrix)
print(cosine_sim)

[[1.         0.         0.         ... 0.         0.         0.11396058]
 [0.         1.         0.10050378 ... 0.11785113 0.12598816 0.        ]
 [0.         0.10050378 1.         ... 0.10660036 0.11396058 0.        ]
 ...
 [0.         0.11785113 0.10660036 ... 1.         0.13363062 0.        ]
 [0.         0.12598816 0.11396058 ... 0.13363062 1.         0.        ]
 [0.11396058 0.         0.         ... 0.         0.         1.        ]]


## Recommendation Function

In [37]:
indices = pd.Series(feature_df.index, index=feature_df['title']).drop_duplicates()

def content_recommender(title):
  if title not in indices:
    return f"{title} was not found in the database."

  idx = indices[title]
  sim_scores = list(enumerate(cosine_sim[idx]))
  sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

  sim_scores = sim_scores[1:11]
  movie_indices = [i[0] for i in sim_scores]

  return base_df.iloc[movie_indices]

In [38]:
def hybrid_recommender(title, top_n=10, similarity_candidates=50, alpha=0.5):
    if title not in indices:
        return f"{title} was not found in the database."

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:similarity_candidates+1]
    movie_indices = [i[0] for i in sim_scores]
    similarity_values = [i[1] for i in sim_scores]

    candidates_df = base_df.iloc[movie_indices].copy()
    candidates_df['similarity'] = similarity_values

    min_score = base_df['score'].min()
    max_score = base_df['score'].max()
    candidates_df['norm_score'] = (candidates_df['score'] - min_score) / (max_score - min_score)

    candidates_df['hybrid_score'] = (
        alpha * candidates_df['similarity'] +
        (1 - alpha) * candidates_df['norm_score']
    )

    hybrid_recommendations = candidates_df.sort_values(by='hybrid_score', ascending=False)

    return hybrid_recommendations.head(top_n)

In [43]:
movie_title = input("Enter the movie title : ")

try:
    rec_df = hybrid_recommender(movie_title, top_n=10, alpha=0.5)
    if isinstance(rec_df, str):
        print(rec_df)
    else:
        display(rec_df[['primaryTitle', 'genres', 'averageRating', 'score', 'numVotes', 'directors', 'writers']])
except Exception as e:
    print(f"Error: {e}")
    print("Movie was not found in the database.")

Enter the movie title : Avatar


,primaryTitle,genres,averageRating,score,numVotes,directors,writers
9236,Alita: Battle Angel,"[Action, Adventure, Sci-Fi]",7.3,7.29,328264,[Robert Rodriguez],"[James Cameron, Laeta Kalogridis, Yukito Kishiro]"
5293,Pirates of the Caribbean: The Curse of the Bla...,"[Action, Adventure, Fantasy]",8.1,8.10,1350045,[Gore Verbinski],"[Ted Elliott, Terry Rossio, Stuart Beattie, Ja..."
80,Star Wars: Episode III - Revenge of the Sith,"[Action, Adventure, Fantasy]",7.7,7.70,936649,[George Lucas],[George Lucas]
4236,Ghosts of the Abyss,"[Documentary, Family]",6.8,6.67,6855,[James Cameron],[]
7379,Pirates of the Caribbean: Dead Man's Chest,"[Action, Adventure, Fantasy]",7.4,7.40,839273,[Gore Verbinski],"[Ted Elliott, Terry Rossio, Stuart Beattie, Ja..."
9712,Wonder Woman,"[Action, Adventure, Fantasy]",7.3,7.30,743346,[Patty Jenkins],"[Allan Heinberg, Zack Snyder, Jason Fuchs, Wil..."
9634,Pirates of the Caribbean: At World's End,"[Action, Adventure, Fantasy]",7.2,7.20,755045,[Gore Verbinski],"[Ted Elliott, Terry Rossio, Stuart Beattie, Ja..."
8296,Hellboy II: The Golden Army,"[Action, Adventure, Fantasy]",7.0,7.00,300940,[Guillermo del Toro],"[Guillermo del Toro, Mike Mignola]"
321,Hellboy,"[Action, Adventure, Fantasy]",6.9,6.90,373954,[Guillermo del Toro],"[Guillermo del Toro, Peter Briggs, Mike Mignola]"
1183,The Mask of Zorro,"[Action, Adventure]",6.4,6.11,21,[],[]
